In [ ]:
#!pip install transformers datasets librosa torch protect accelerate evaluate seqeval

In [ ]:
#!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/colab_packages')

### for importing evaluate

In [ ]:
import os
import gc
import json
import torch
import pandas as pd
import itertools
import evaluate
import numpy as np
from pathlib import Path
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Trainer, TrainingArguments, Wav2Vec2Processor
)

In [ ]:
base_dir = Path("/content/drive/MyDrive/Colab022Notebooks/GP")
arrow_save_dir = base_dir / "Arrow_Datasets"
models_save_dir = base_dir / "Saved_Trained_Models"
results_dir = base_dir / "Master_Grid_Search_Results"

os.makedirs(models_save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Models & hyperparamters
models = ["facebook/wav2vec2-base", "facebook/hubert-base-ls960"]
learning_rates = [3e-5, 5e-5]
batch_sizes = [8, 16]

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

# Data collator
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
@dataclass
class DataCollatorForAudio:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [feature["label"] for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor(label_features)
        return batch

dummy_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
data_collator = DataCollatorForAudio(processor=dummy_processor)

master_results = pd.DataFrame(columns=['Dataset', 'Model', 'LR', 'Batch_Size', 'Accuracy', 'Loss', 'Model_Path'])
datasets_available = ["CREMA-D", "EYASE"]


In [ ]:

print("🚀 STARTING TRAINING PIPELINE...")

for dataset_name in datasets_available:
    dataset_path = arrow_save_dir / dataset_name
    label_path = arrow_save_dir / f"{dataset_name}_labels.json"

    if not dataset_path.exists():
        continue

    print(f"\n{'='*50}\n📂 LOADING ARROW DATASET: {dataset_name}\n{'='*50}")

    # fetch Data
    encoded_dataset = load_from_disk(str(dataset_path))
    encoded_train = encoded_dataset["train"]
    encoded_test = encoded_dataset["test"]

    # 2. fetch labels
    with open(label_path, "r") as f:
        label2id = json.load(f)
    id2label = {int(v): k for k, v in label2id.items()}
    num_labels = len(label2id)

    combinations = list(itertools.product(models, learning_rates, batch_sizes))

    for model_checkpoint, lr, bs in combinations:
        clean_model_name = model_checkpoint.split('/')[-1]
        exp_name = f"{dataset_name}_{clean_model_name}_LR{lr}_BS{bs}"
        final_model_path = models_save_dir / exp_name

        print(f"\n🔥 Training: {exp_name}")

        torch.cuda.empty_cache()
        gc.collect()

        if "hubert" in model_checkpoint:
            model = HubertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label)
        else:
            model = Wav2Vec2ForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label)

        model.freeze_feature_encoder()

        training_args = TrainingArguments(
            output_dir=f"{str(base_dir/"ModelExp")}_{exp_name}",
            eval_strategy="epoch",
            save_strategy="epoch",            # save every Epoch
            load_best_model_at_end=True,      # return best Epoch
            metric_for_best_model="accuracy",
            learning_rate=lr,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs,
            num_train_epochs=3,
            weight_decay=0.01,
            fp16=True,
            save_total_limit=1,
            remove_unused_columns=False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=encoded_train,
            eval_dataset=encoded_test,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        eval_result = trainer.evaluate()

        # save models
        trainer.save_model(str(final_model_path))
        print(f"💾 Model Saved to: {final_model_path}")

        new_row = {
            'Dataset': dataset_name,
            'Model': clean_model_name,
            'LR': lr,
            'Batch_Size': bs,
            'Accuracy': eval_result['eval_accuracy'],
            'Loss': eval_result['eval_loss'],
            'Model_Path': str(final_model_path)
        }
        master_results = pd.concat([master_results, pd.DataFrame([new_row])], ignore_index=True)
        master_results.to_csv(results_dir / "final_grid_search_report.csv", index=False)

print("\n🎉 ALL PIPELINES EXECUTED FLAWLESSLY! 🎉")

🚀 STARTING TRAINING PIPELINE...

📂 LOADING ARROW DATASET: CREMA-D

🔥 Training: CREMA-D_wav2vec2-base_LR3e-05_BS8


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model Saved to: /content/drive/MyDrive/Colab022Notebooks/GP/Saved_Trained_Models/TESS_wav2vec2-base_LR5e-05_BS16

🔥 Training: TESS_hubert-base-ls960_LR3e-05_BS8


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

HubertForSequenceClassification LOAD REPORT from: facebook/hubert-base-ls960
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 
projector.weight  | MISSING | 
projector.bias    | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


##**Resuming Strategy**

In [ ]:
import os
import gc
import json
import torch
import pandas as pd
import itertools
import evaluate
import numpy as np
from pathlib import Path
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Trainer, TrainingArguments, Wav2Vec2Processor
)

# ==========================================
# 1. paths
# ==========================================
out_dir = Path("/content/drive/MyDrive/Colab022Notebooks/GP")
arrow_save_dir = out_dir / "Arrow_Datasets"
models_save_dir = out_dir / "Saved_Trained_Models"
results_dir = out_dir / "Master_Grid_Search_Results"

os.makedirs(models_save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# ==========================================
# 2. (The Resume Logic)
# ==========================================
csv_path = results_dir / "final_grid_search_report.csv"

# if csv paht exists , read and continue
if csv_path.exists():
    print("📥 Loading existing Grid Search Report to resume...")
    master_results = pd.read_csv(csv_path)
else:
    print("🆕 Creating new Grid Search Report...")
    master_results = pd.DataFrame(columns=['Dataset', 'Model', 'LR', 'Batch_Size', 'Accuracy', 'Loss', 'Model_Path'])

models = ["facebook/wav2vec2-base", "facebook/hubert-base-ls960"]
learning_rates = [3e-5, 5e-5]
batch_sizes = [8, 16]

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
@dataclass
class DataCollatorForAudio:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [feature["label"] for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor(label_features)
        return batch

dummy_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
data_collator = DataCollatorForAudio(processor=dummy_processor)

datasets_available = ["CREMA-D", "EYASE", "TESS", "BAVED"]

print("🚀 STARTING RESUME PIPELINE...")

# ==========================================
# 3. Smart retrain
# ==========================================
for dataset_name in datasets_available:
    dataset_path = arrow_save_dir / dataset_name
    label_path = arrow_save_dir / f"{dataset_name}_labels.json"

    if not dataset_path.exists():
        print(f"⚠️ Warning: Dataset {dataset_name} not found. Skipping...")
        continue

    print(f"\n{'='*50}\n📂 DATASET: {dataset_name}\n{'='*50}")

    encoded_dataset = load_from_disk(str(dataset_path))
    encoded_train = encoded_dataset["train"]
    encoded_test = encoded_dataset["test"]

    with open(label_path, "r") as f:
        label2id = json.load(f)
    id2label = {int(v): k for k, v in label2id.items()}
    num_labels = len(label2id)

    combinations = list(itertools.product(models, learning_rates, batch_sizes))

    for model_checkpoint, lr, bs in combinations:
        clean_model_name = model_checkpoint.split('/')[-1]
        exp_name = f"{dataset_name}_{clean_model_name}_LR{lr}_BS{bs}"
        final_model_path = models_save_dir / exp_name

        # ----------------------------------------------------
        # if trained , skip
        # ----------------------------------------------------
        if final_model_path.exists():
            print(f"⏭️ SKIPPING: {exp_name} (Already Trained & Saved)")
            continue

        print(f"\n🔥 Training: {exp_name}")

        torch.cuda.empty_cache()
        gc.collect()

        if "hubert" in model_checkpoint:
            model = HubertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label)
        else:
            model = Wav2Vec2ForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label)

        model.freeze_feature_encoder()

        temp_out_dir = out_dir / f"temp_{exp_name}"

        training_args = TrainingArguments(
            output_dir=str(temp_out_dir),
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            learning_rate=lr,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs,
            num_train_epochs=3,
            weight_decay=0.01,
            fp16=True,
            save_total_limit=1,
            remove_unused_columns=False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=encoded_train,
            eval_dataset=encoded_test,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        # search for checkpoint
        last_checkpoint = None
        if temp_out_dir.exists():
            checkpoints = [d for d in os.listdir(temp_out_dir) if d.startswith("checkpoint-")]
            if checkpoints:
                # select the recent checkpoint
                checkpoints.sort(key=lambda x: int(x.split('-')[1]))
                last_checkpoint = str(temp_out_dir / checkpoints[-1])
                print(f"🔄 Found checkpoint! Resuming from: {last_checkpoint}")

        # train
        trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)
        eval_result = trainer.evaluate()

        # save
        trainer.save_model(str(final_model_path))
        print(f"💾 Model Saved to: {final_model_path}")

        new_row = {
            'Dataset': dataset_name,
            'Model': clean_model_name,
            'LR': lr,
            'Batch_Size': bs,
            'Accuracy': eval_result['eval_accuracy'],
            'Loss': eval_result['eval_loss'],
            'Model_Path': str(final_model_path)
        }

        new_df = pd.DataFrame([new_row])
        master_results = pd.concat([master_results, new_df], ignore_index=True)
        master_results.to_csv(csv_path, index=False)

print("\n🎉 ALL PIPELINES EXECUTED FLAWLESSLY! 🎉")

📥 Loading existing Grid Search Report to resume...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model Saved to: /content/drive/MyDrive/Colab022Notebooks/GP/Saved_Trained_Models/BAVED_hubert-base-ls960_LR5e-05_BS16

🎉 ALL PIPELINES EXECUTED FLAWLESSLY! 🎉
